# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: c7a2f744-5446-47ce-bdc3-5aa1d06dc6ef
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session c7a2f744-5446-47ce-bdc3-5aa1d06dc6ef to get into ready status...
Session c7a2f744-5446-47ce-bdc3-5aa1d06dc6ef 

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [4]:
# Leitura da camada Gold consolidada no Amazon S3

gold_path = "s3://fiap-tech-challenge-fase3-vicagt/gold/base_consolidada/"

df_gold = spark.read.parquet(gold_path)

# Consulta analítica básica da camada Gold
total_registros = df_gold.count()
total_colunas = len(df_gold.columns)

print("=== CONSULTA ANALÍTICA - CAMADA GOLD ===")
print(f"Total de registros: {total_registros}")
print(f"Total de colunas: {total_colunas}")

=== CONSULTA ANALÍTICA - CAMADA GOLD ===
Total de registros: 14002
Total de colunas: 1194


In [5]:
# Identificação das colunas relacionadas ao período da pesquisa

colunas_periodo = [
    coluna for coluna in df_gold.columns
    if "period" in coluna.lower()
    or "ano" in coluna.lower()
    or "year" in coluna.lower()
]

print("=== COLUNAS RELACIONADAS A PERÍODO ===")

for coluna in colunas_periodo:
    print(coluna)

=== COLUNAS RELACIONADAS A PERÍODO ===
2_n_planos_de_mudar_de_emprego_6m
2_o_7_plano_de_carreira_e_oportunidades_de_crescimento
8_b_4_utilizo_metodos_estatisticos_bayesianos_para_analisar_dados
col_2_n_planos_de_mudar_de_emprego_6m
col_2_o_7_plano_de_carreira_e_oportunidades_de_crescimento
col_8_b_4_utilizo_metodos_estatisticos_bayesianos_para_analisar_dados
p2_o_7_plano_de_carreira_e_oportunidades_de_crescimento_profissional
p8_b_4_utilizo_metodos_estatisticos_bayesianos_para_analisar_dados
periodo


In [6]:
# Consulta analítica: distribuição de registros por período da pesquisa

from pyspark.sql import functions as F

analise_periodo = (
    df_gold
    .groupBy("periodo")
    .agg(F.count("*").alias("total_registros"))
    .orderBy("periodo")
)

print("=== DISTRIBUIÇÃO DE REGISTROS POR PERÍODO ===")
analise_periodo.show(truncate=False)

=== DISTRIBUIÇÃO DE REGISTROS POR PERÍODO ===
+---------+---------------+
|periodo  |total_registros|
+---------+---------------+
|2023-2024|5293           |
|2024-2025|5215           |
|2025-2026|3494           |
+---------+---------------+


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

In [7]:
# Consulta analítica: diagnóstico da camada Gold

print("=== DIAGNÓSTICO ANALÍTICO DA CAMADA GOLD ===")
print(f"Total de registros: {df_gold.count()}")
print(f"Total de colunas: {len(df_gold.columns)}")
print(f"Períodos disponíveis: {df_gold.select('periodo').distinct().count()}")

print("Períodos analisados:")
df_gold.select("periodo").distinct().orderBy("periodo").show(truncate=False)

=== DIAGNÓSTICO ANALÍTICO DA CAMADA GOLD ===
Total de registros: 14002
Total de colunas: 1194
Períodos disponíveis: 3
Períodos analisados:
+---------+
|periodo  |
+---------+
|2023-2024|
|2024-2025|
|2025-2026|
+---------+


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)